In [ ]:
import geopandas as gpd
import pandas as pd
from pathlib import Path
import sys
from loguru import logger
import arrow

sys.path.append("..")
from data import get_config

data_path = Path("../data")

logger.remove()
logger.add(sys.stderr, level="INFO")

: 

In [7]:
config = {
	"Full": get_config("Full"),
	"Simple": get_config("Simple"),
	"DiGiorgio": get_config("DiGiorgio"),
}

In [4]:
gpd.read_file(
	r"G:\Arvin-Edison WSD-1215\121523003-Frick Unit Pipeline\300 CAD\320 References\Sites_Ref\Shapefiles\Pipenew.shp"
).head(2)

,NAME,geometry
0,"4""","LINESTRING Z (6291607.387 2304596.408 0, 62916..."
1,"8""","LINESTRING Z (6291607.308 2304585.859 0, 62915..."


In [ ]:
district_boundary = gpd.read_file(
    config['Full'].loc[config['Full']["Name"] == "District Boundary"]["file_path"].iloc[0]
)
service_boundary = gpd.read_file(
    config['Full'].loc[config['Full']["Name"] == "Frick Unit Service Area"]["file_path"].iloc[
        0
    ]
)

intersect = lambda gdf: gdf[gdf.intersects(service_boundary.unary_union)]
clip = lambda gdf: gdf.clip(service_boundary)


def get_layer(row, sheet_name):
    try:
        logger.info(f"Loading layer {row['Name']}")
        file_path = Path(row["file_path"])
        read_options = {
            ".shp": lambda file: gpd.read_file(file),
            ".gdb": lambda file: gpd.read_file(file, layer=row["layer"]),
            ".parquet": lambda file: gpd.read_parquet(file),
        }
        # logger.info(file_path)
        gdf = read_options[file_path.suffix](file_path).to_crs(epsg=2229)

        if row["Name"] == "APNs":
            if sheet_name == "Full":
                gdf["label"] = (
                    gdf["label"] + "\nService Area Type: " + gdf["Service Area Type"]
                )
            gdf["label"] = gdf["label"].str.replace(r"\n", "<br>", regex=True)

            # gdf = full_gdf[full_gdf['layer']==layer]
        if row["Name"] == "Panama Unit Service Area":
            gdf = gdf.loc[gdf["Name"] != "Frick Unit North Service Area"]

        if row["Name"] == "Panama Unit Pipeline":
            gdf = gdf.loc[gdf["Name"] != "Frick Unit"]

        gdf["color"] = row["color"]
        if type(row["label"]) == str:
            gdf["label"] = gdf[row["label"]].astype(str).replace("nan", "")
        else:
            gdf["label"] = ""

        gdf["layer"] = row["Name"]
        gdf["size"] = row["size"]
        if row["clip_to_unit"] == True:
            logger.info(f"Clipping layer {row['Name']}")
            gdf = clip(gdf)
        logger.info(list(gdf.columns))

        return gdf[
            ["color", "label", "layer", "size", "geometry"]
        ]  # .to_crs(epsg=4326)
    except Exception as e:
        logger.error(f"Layer {row['Name']} failed to load due to {e}")
        logger.error(gdf.columns)
        return None

In [9]:
def get_sheet(sheet_name):
    gdfs = {y["Name"]: get_layer(y, sheet_name) for i, y in config[sheet_name].iterrows()}
    epsg = 4326
    gdf = pd.concat([gdf.to_crs(epsg=epsg) for gdf in gdfs.values() if gdf is not None])
    gdf.to_parquet(
        data_path / f"{sheet_name}-{arrow.now().format('YYYY-MM-DD')}.parquet"
    )


get_sheet("Full")
get_sheet("Simple")
get_sheet("DiGiorgio")

2025-12-31 08:15:47.937 | INFO     | __main__:get_layer:16 - Loading layer FFPPP Discharge Pipeline
2025-12-31 08:15:48.182 | INFO     | __main__:get_layer:53 - ['ID', 'CANAL', 'Nitrogen', 'length', 'geometry', 'color', 'label', 'layer', 'size']
2025-12-31 08:15:48.183 | INFO     | __main__:get_layer:16 - Loading layer AEWSD North Canal
2025-12-31 08:15:48.203 | INFO     | __main__:get_layer:53 - ['ID', 'CANAL', 'Nitrogen', 'length', 'geometry', 'color', 'label', 'layer', 'size']
2025-12-31 08:15:48.205 | INFO     | __main__:get_layer:16 - Loading layer District Boundary
2025-12-31 08:15:48.229 | INFO     | __main__:get_layer:53 - ['AREA', 'PERIMETER', 'DIV_FRIANT', 'WDNAME', 'SHADESYM', 'PNAME', 'IDCON', 'DIV_FRIA_1', 'ACRES', 'HECTARES', 'CalcArea', 'TotalAcres', 'geometry', 'color', 'label', 'layer', 'size']
2025-12-31 08:15:48.231 | INFO     | __main__:get_layer:16 - Loading layer Sandrini Unit Service Area
2025-12-31 08:15:48.385 | INFO     | __main__:get_layer:53 - ['OID_', 'Name

KeyboardInterrupt: 